## Step 1: Setup and Installation

In [2]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [3]:
   import os
   import shutil
   from dotenv import load_dotenv

In [5]:
import os
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
SAMPLE_DOCS_DIR = os.path.join(os.getcwd(), "sample_docs")

print("OpenAI key:   loaded")
print("Node.js npx: ", "found" if shutil.which("npx") else "NOT found")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing")
print("sample_docs: ", "found" if os.path.isdir(SAMPLE_DOCS_DIR) else "NOT found")
print("Folder used:  ", SAMPLE_DOCS_DIR)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM ready")

OpenAI key:   loaded
Node.js npx:  found
GitHub token: loaded
sample_docs:  found
Folder used:   c:\Users\bbars\Desktop\Ironhack AI Integration & Consulting\Week 4\Day 3\Lab\lab-mcp-langchain\sample_docs
LLM ready


In [7]:
import sys

# Fixes a Windows + Jupyter problem when starting a local (stdio) MCP server.
# It sends the server's error output to "nowhere" so the connection doesn't crash.
if sys.platform == "win32":
    import mcp.client.stdio as _stdio
    _stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

print("Windows fix applied" if sys.platform == "win32" else "Not Windows, no fix needed")

Windows fix applied


In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE the tool runs: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")

    result = await handler(request)   # the real tool runs on the server

    # AFTER the tool runs: what the server sent back (first 150 characters)
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result

print("Interceptor ready")

Interceptor ready


## Step 2: Connect to MCP Server

Step 2a: Connect to the filesystem server (local)

In [9]:
filesystem_client = MultiServerMCPClient(
    {
        "filesystem": {                      # name of this server
            "transport": "stdio",            # local: the client starts the server
            "command": "npx",                # the program to run
            "args": [
                "-y",                                        # auto-confirm the download
                "@modelcontextprotocol/server-filesystem",   # the server package
                SAMPLE_DOCS_DIR,                             # the ONLY folder it may see
            ],
        }
    },
    tool_interceptors=[log_tool_calls],      # plug in our logger
)

print("Filesystem client created")

Filesystem client created


Step 2b: Connect to the GitHub server (remote)

In [10]:
github_client = MultiServerMCPClient(
    {
        "github": {                                        # name of this server
            "transport": "streamable_http",                # remote: talk over the internet
            "url": "https://api.githubcopilot.com/mcp/",   # GitHub's official MCP server
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}", # your "ID card"
                "X-MCP-Toolsets": "repos,issues",          # only load these tool groups
            },
        }
    },
    tool_interceptors=[log_tool_calls],                    # same logger as before
)

print("GitHub client created")

GitHub client created


Step 2c: Test both connections

In [11]:
clients = {
    "filesystem": filesystem_client,
    "github": github_client,
}

for name, client in clients.items():
    # 1. Tools
    tools = await client.get_tools()
    print(f"{name}: {len(tools)} tools")

    # 2. Resources (not every server has them)
    try:
        resources = await client.get_resources()
        if resources:
            print(f"{name}: {len(resources)} resources")
        else:
            print(f"{name}: no resources")
    except Exception as e:
        print(f"{name}: no resources ({type(e).__name__})")

    print("-" * 30)

filesystem: 14 tools
filesystem: no resources (ExceptionGroup)
------------------------------
github: 29 tools
github: 4 resources
------------------------------


Step 3: Load MCP tools into LangChain

In [12]:
# Load the tools from each client and save them
fs_tools = await filesystem_client.get_tools()
github_tools = await github_client.get_tools()

# Print the name of every tool
print(f"Filesystem tools ({len(fs_tools)}):")
for tool in fs_tools:
    print("  -", tool.name)

print(f"\nGitHub tools ({len(github_tools)}):")
for tool in github_tools:
    print("  -", tool.name)

Filesystem tools (14):
  - read_file
  - read_text_file
  - read_media_file
  - read_multiple_files
  - write_file
  - edit_file
  - create_directory
  - list_directory
  - list_directory_with_sizes
  - directory_tree
  - move_file
  - search_files
  - get_file_info
  - list_allowed_directories

GitHub tools (29):
  - add_issue_comment
  - create_branch
  - create_or_update_file
  - create_repository
  - delete_file
  - fork_repository
  - get_commit
  - get_file_contents
  - get_label
  - get_latest_release
  - get_release_by_tag
  - get_tag
  - issue_read
  - issue_write
  - list_branches
  - list_commits
  - list_issue_fields
  - list_issue_types
  - list_issues
  - list_releases
  - list_repository_collaborators
  - list_tags
  - push_files
  - search_code
  - search_commits
  - search_issues
  - search_repositories
  - sub_issue_write
  - update_issue_comment


Step 4a: Create the Filesystem agent

In [13]:
fs_agent = create_agent(
    model=llm,
    tools=fs_tools,
    system_prompt=f"""You are a document assistant.
You can read files ONLY in this folder: {SAMPLE_DOCS_DIR}

Rules:
1. First, list the folder to see which files exist.
2. Then read the files you need.
3. Never guess. If the answer is not in the files, say "I could not find this in the files."
4. Always name the file(s) you used for your answer.""",
)

question = "Who is the engagement lead for Project Atlas, and what is the target go-live date?"

fs_result = await fs_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("\nFINAL ANSWER:")
print(fs_result["messages"][-1].content)

[LOG] filesystem -> list_directory  {'path': 'c:\\Users\\bbars\\Desktop\\Ironhack AI Integration & Consulting\\Week 4\\Day 3\\Lab\\lab-mcp-langchain\\sample_docs'}
[LOG]    returned: [FILE] faq.md
[FILE] project_notes.md...
[LOG] filesystem -> read_text_file  {'path': 'c:\\Users\\bbars\\Desktop\\Ironhack AI Integration & Consulting\\Week 4\\Day 3\\Lab\\lab-mcp-langchain\\sample_docs\\faq.md'}
[LOG] filesystem -> read_text_file  {'path': 'c:\\Users\\bbars\\Desktop\\Ironhack AI Integration & Consulting\\Week 4\\Day 3\\Lab\\lab-mcp-langchain\\sample_docs\\project_notes.md'}
[LOG]    returned: # Project Atlas – Project Notes

## Overview
Project Atlas is a support-ticket assistant for our client, Nordlicht Logistics GmbH.
The assistant reads...
[LOG]    returned: # Project Atlas – FAQ

**Q: Who do I contact with questions about the project?**
A: Maria Keller, the engagement lead. For technical questions, contac...

FINAL ANSWER:
The engagement lead for Project Atlas is Maria Keller, and th

Step 4b: Create the GitHub agent

In [14]:
github_agent = create_agent(
    model=llm,
    tools=github_tools,
    system_prompt="""You are a GitHub research assistant.
You read GitHub repositories ONLY with your GitHub tools.

Rules:
1. To read a file, use get_file_contents with the owner, repo and file path.
2. Never guess. If the answer is not in the file, say "I could not find this in the repository."
3. Always name the repository and the file you used.""",
)

question = (
    "Read the README of the GitHub repository langchain-ai/langchain-mcp-adapters. "
    "Which MCP transports does it show in its examples?"
)

github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("\nFINAL ANSWER:")
print(github_result["messages"][-1].content)

[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langchain-mcp-adapters', 'path': 'README.md'}
[LOG]    returned: successfully downloaded text file (SHA: 0d3e7d0fc7c4facdf437445eaeeb71102de4befe) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

FINAL ANSWER:
The README of the repository `langchain-ai/langchain-mcp-adapters` shows examples of the following MCP transports:

1. **stdio** - Used for local server communication.
2. **http** - Used for connecting to MCP servers over HTTP.
3. **streamable_http** - A transport that supports streamable HTTP connections.

These transports are demonstrated in various examples throughout the README.


Step 5: Access MCP resources

In [15]:
# 1. Connect to the LangChain docs server (remote, no token needed)
docs_client = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp",
        }
    }
)

# 2. Load the resources and read the first one as text
resources = await docs_client.get_resources("langchain-docs")
print(f"Found {len(resources)} resources")

guide_text = resources[0].as_string()
print(f"First resource: {len(guide_text)} characters")
print("Preview:", guide_text[:200], "...\n")

# 3. Agent with NO tools, the guide goes into the system prompt
guide_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=f"""Answer ONLY using the guide below.
If the answer is not in the guide, say "This is not in the guide."

GUIDE:
{guide_text}""",
)

question = "What do I need to add to an agent so that it remembers the conversation between calls?"

guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content

print("FINAL ANSWER:")
print(answer)

# 4. Evidence: is a key word from the answer really in the guide?
keyword = "thread_id"
print(f"\nEvidence check: '{keyword}' in guide? ->", keyword in guide_text)

Found 1 resources
First resource: 11252 characters
Preview: ---
name: langchain
description: Use when building AI agents, integrating language models with
  tools, orchestrating multi-step workflows, or deploying production agent
  systems. Reach for this skil ...

FINAL ANSWER:
You need to pass `config={"configurable": {"thread_id": "..."}}` in the agent invocation to persist conversation history across calls.

Evidence check: 'thread_id' in guide? -> True


In [16]:
print("'checkpointer' in guide? ->", "checkpointer" in guide_text)

'checkpointer' in guide? -> True


Step 6: One agent, two servers

In [17]:
all_tools = fs_tools + github_tools

print(f"Filesystem tools: {len(fs_tools)}")
print(f"GitHub tools:     {len(github_tools)}")
print(f"Total:            {len(all_tools)}")
print("\nAll tool names:")
print([tool.name for tool in all_tools])

Filesystem tools: 14
GitHub tools:     29
Total:            43

All tool names:
['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'add_issue_comment', 'create_branch', 'create_or_update_file', 'create_repository', 'delete_file', 'fork_repository', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'issue_write', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'push_files', 'search_code', 'search_commits', 'search_issues', 'search_repositories', 'sub_issue_write', 'update_issue_comment']


In [18]:
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=f"""You are a research assistant with two sources:

1. LOCAL FILES: read them with the filesystem tools.
   You may only access this folder: {SAMPLE_DOCS_DIR}
   List the folder first before reading files.

2. GITHUB: read repositories with the GitHub tools
   (for example get_file_contents with owner, repo and path).

Rules:
- Never guess. If something is not in the sources, say so.
- For every part of your answer, name the source (file name, or repository + file).""",
)

question = (
    "Read our Project Atlas notes (project_notes.md): what constraint do they set about where ticket data may live? "
    "Then read the README of the GitHub repository langchain-ai/langchain-mcp-adapters: "
    "which transport would let us run an MCP server inside the client's own environment, "
    "so that data never leaves it?"
)

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})

print("\nFINAL ANSWER:")
print(combined_result["messages"][-1].content)

[LOG] filesystem -> read_text_file  {'path': 'c:\\Users\\bbars\\Desktop\\Ironhack AI Integration & Consulting\\Week 4\\Day 3\\Lab\\lab-mcp-langchain\\sample_docs\\project_notes.md'}
[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langchain-mcp-adapters', 'path': 'README.md'}
[LOG]    returned: # Project Atlas – Project Notes

## Overview
Project Atlas is a support-ticket assistant for our client, Nordlicht Logistics GmbH.
The assistant reads...
[LOG]    returned: successfully downloaded text file (SHA: 0d3e7d0fc7c4facdf437445eaeeb71102de4befe) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

FINAL ANSWER:
### Project Atlas Notes (project_notes.md)

The constraint set about where ticket data may live is as follows:
- **All ticket data must stay inside the client's own environment. No ticket data may be sent to or stored on external servers.**

### GitHub Repository (langchain-ai/langchain-mcp-adapters)

In the README of the GitHub repository

In [19]:
my_question = (
    "According to faq.md, which languages does the Project Atlas assistant support? "
    "Also, what is the latest release of the GitHub repository langchain-ai/langchain-mcp-adapters?"
)

my_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=my_question)]})

print("\nFINAL ANSWER:")
print(my_result["messages"][-1].content)

[LOG] filesystem -> read_text_file  {'path': 'c:\\Users\\bbars\\Desktop\\Ironhack AI Integration & Consulting\\Week 4\\Day 3\\Lab\\lab-mcp-langchain\\sample_docs\\faq.md'}
[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langchain-mcp-adapters'}
[LOG]    returned: # Project Atlas – FAQ

**Q: Who do I contact with questions about the project?**
A: Maria Keller, the engagement lead. For technical questions, contac...
[LOG]    returned: {"tag_name":"langchain-mcp-adapters==0.3.2","target_commitish":"main","name":"langchain-mcp-adapters==0.3.2","body":"## What's Changed\n* build(deps):...

FINAL ANSWER:
According to the **faq.md** file, the Project Atlas assistant supports the following languages: **German and English**.

As for the latest release of the GitHub repository **langchain-ai/langchain-mcp-adapters**, it is version **0.3.2**. You can find more details about this release [here](https://github.com/langchain-ai/langchain-mcp-adapters/releases/tag/langchain-mcp